# Exercise 8 (Optional) — Full Waveform Inversion (FWI) with JUDI

This notebook is a **single-vintage** version of the nonlinear time-lapse inversion notebook, rewritten as a class exercise on **acoustic FWI**.

## Learning goals
By the end of this exercise, you should be able to:
- generate synthetic seismic data from a velocity model,
- build a smooth starting model,
- define an FWI objective and gradient with `judiJacobian`,
- run a few iterations of nonlinear inversion,
- compare the recovered model with the true model.

## What you should notice
- FWI is a **nonlinear** inverse problem.
- The starting model matters.
- Low frequencies and a smooth initial model help convergence.
- The adjoint-state gradient often highlights reflectors first, not the full low-wavenumber model.

## 1. Compute environment

Set environment variables for the Devito backend used by JUDI.

In [ ]:
# Unix gcc config
ENV["DEVITO_LOGGING"] = "ERROR"

## 2. Package setup

The install cell can be skipped after the first run.

In [ ]:
# using Pkg
# Pkg.add(["SlimOptim"])

In [ ]:
using JUDI, SlimOptim, SlimPlotting, JLD2, PyPlot, Random, LinearAlgebra, Images

## 3. Load and prepare the model

We treat the baseline model as the **true model** and build a smooth background model for inversion.

In [ ]:
if !isfile("CompassTimeLapse.jld2")
    download(
        "https://github.com/slimgroup/Software.SEG2021/raw/main/CompassTimeLapseCCS.jl/model/CompassTimeLapse.jld2",
        "CompassTimeLapse.jld2"
    )
end

In [ ]:
JLD2.@load "CompassTimeLapse.jld2" n d o m_stack

In [ ]:
# Coarsen the grid for a classroom-sized FWI example
d = (15f0, 15f0)

# Use one vintage only
m_true = convert(Matrix{Float32}, m_stack[1][1:4:end, 1:2:end])

# Smooth initial model
m0 = copy(m_true)
m0[:, 12:end] = imfilter(m0[:, 12:end].^(0.5f0), Kernel.gaussian(5f0)).^2
m0 = convert(Matrix{Float32}, m0)

# Water bottom, used for receiver depth
wb = find_water_bottom(m_true .- maximum(m_true))

# Updated size after subsampling
n = size(m_true)
n, d, o

### Plot the true and starting models

In [ ]:
figure(figsize=(12, 8))
subplot(211)
plot_velocity(m_true', d; new_fig=false, name="True model", cmap="cet_rainbow4_r", cbar=true)
subplot(212)
plot_velocity(m0', d; new_fig=false, name="Smooth starting model", cmap="cet_rainbow4_r", cbar=true)
tight_layout()

### Question 1
Why is the starting model smoother than the true model?

## 4. Acquisition geometry and source wavelet

We define a simple ocean-bottom acquisition:
- sources near the surface,
- receivers near the seafloor,
- a low-frequency source wavelet to make inversion easier.

In [ ]:
# Recording parameters
timeD = 3000f0      # ms
dtD   = 2f0         # ms
f0    = 0.0145f0    # 14.5 Hz in kHz units used by JUDI
wavelet = low_filter(ricker_wavelet(timeD, dtD, f0), 4f0; fmin=3, fmax=15)

In [ ]:
# Source coordinates
xsrc = convertToCell(range(start=30f0, step=30f0, stop=(n[1]-1)*d[1]-30f0))
nsrc = length(xsrc)
ysrc = convertToCell(range(0f0, stop=0f0, length=nsrc))
zsrc = convertToCell(range(d[1], stop=d[1], length=nsrc))

srcGeometry = Geometry(xsrc, ysrc, zsrc; dt=dtD, t=timeD)

In [ ]:
# Receiver coordinates
xrec = range(start=60f0, step=60f0, stop=(n[1]-1)*d[1]-60f0)
yrec = 0f0
zrec = (wb[5:4:end-5] .- 1) .* d[2]

recGeometry = Geometry(xrec, yrec, zrec; dt=dtD, t=timeD, nsrc=nsrc)

In [ ]:
figure(figsize=(15, 5))
plot_velocity(m_true', d; new_fig=false, name="Acquisition geometry", cmap="cet_rainbow4_r", cbar=true)
scatter(xsrc[1:5:end], zsrc[1:5:end], c="r", label="Sources")
scatter(xrec, zrec, c="c", label="Receivers")
legend(loc="lower left")

## 5. JUDI setup and synthetic data generation

We now:
1. create the JUDI source vector,
2. define the true and starting models,
3. generate observed data from the true model.

In [ ]:
q = judiVector(srcGeometry, wavelet)

model_true = Model(n, d, o, m_true; nb=80)
model0     = Model(n, d, o, m0; nb=80)

opt = Options(dt_comp=1f0)
F_true = judiModeling(model_true, srcGeometry, recGeometry; options=opt)
F0     = judiModeling(model0, srcGeometry, recGeometry; options=opt)

In [ ]:
if isfile("d_fwi_true.jld2")
    @load "d_fwi_true.jld2" d_obs
else
    d_obs = F_true * q
    @save "d_fwi_true.jld2" d_obs
end

In [ ]:
figure(figsize=(16, 5))
shot_id = div(nsrc, 2)
plot_sdata(d_obs[shot_id]; new_fig=false, cmap="PuOr", name="Observed data for one shot", cbar=true)

### Question 2
What features of the shot record do you expect to be most sensitive to the shallow part of the model?

## 6. Define the FWI objective

We solve the nonlinear least-squares problem

\[
\phi(m) = \frac{1}{2}\|F(m)q - d_{obs}\|_2^2
\]

At each iteration:
- we model synthetic data,
- compute the residual,
- form the Jacobian at the current model,
- apply the adjoint Jacobian to get the gradient.

In [ ]:
batchsize = 12
N = prod(n)

In [ ]:
function fwi_objective(mvec)
    # Random source batch
    inds = randperm(nsrc)[1:batchsize]

    # Forward model at current iterate
    d_pred = F0(; m=mvec)[inds] * q[inds]

    # Residual
    r = d_pred - d_obs[inds]

    # Jacobian at current model
    J = judiJacobian(F0(; m=mvec), q)

    # Gradient
    g = J[inds]' * r

    # Misfit
    f = 0.5f0 * norm(r)^2

    return f, vec(g)
end

### Compute and inspect the first gradient

In [ ]:
f0, g0 = fwi_objective(vec(m0))
f0

In [ ]:
figure(figsize=(12, 8))
subplot(211)
plot_velocity(m0', d; new_fig=false, name="Starting model", cmap="cet_rainbow4_r", cbar=true)
subplot(212)
plot_simage(reshape(g0, n)', d; new_fig=false, name="First FWI gradient", cmap="seismic", cbar=true, perc=99)
tight_layout()

### Question 3
Does the first gradient look more like a smooth velocity correction or more like a reflector image? Why?

## 7. Add simple bound constraints

We use a projected quasi-Newton method (`pqn`) with box constraints so the model stays within a reasonable velocity range.

In [ ]:
m_min = minimum(m_true)
m_max = maximum(m_true)

constraints = Vector{set_definitions}()
push!(constraints, set_definitions(
    "bounds", "identity", m_min, m_max, ("matrix",""), ([], false)
))

prj = setup_constraints(constraints, vec(m0), Inf, 1)

## 8. Run a few FWI iterations

For a class exercise, we keep the number of iterations small.

In [ ]:
niter = 10
pqn_opt = pqn_options(verbose=3, maxIter=niter, memory=3, corrections=10)

Random.seed!(1234)
sol = pqn(fwi_objective, vec(m0), prj, pqn_opt)

## 9. Compare results

In [ ]:
m_inv = reshape(sol.x, n)

In [ ]:
figure(figsize=(12, 12))
subplot(311)
plot_velocity(m_true', d; new_fig=false, name="True model", cmap="cet_rainbow4_r", cbar=true)
subplot(312)
plot_velocity(m0', d; new_fig=false, name="Starting model", cmap="cet_rainbow4_r", cbar=true)
subplot(313)
plot_velocity(m_inv', d; new_fig=false, name="Inverted model", cmap="cet_rainbow4_r", cbar=true)
tight_layout()

In [ ]:
figure(figsize=(12, 10))
subplot(211)
plot_simage((m_true - m0)', d; new_fig=false, name="True update relative to start", cmap="seismic", cbar=true, perc=99)
subplot(212)
plot_simage((m_inv - m0)', d; new_fig=false, name="Recovered update relative to start", cmap="seismic", cbar=true, perc=99)
tight_layout()

### Question 4
Compare the true update and recovered update:
- where does inversion improve the model most?
- where does it struggle?
- what would you try next to improve the result?

## 10. Optional exploration

Try one change at a time and note what happens:

1. Increase `niter` from 10 to 20.
2. Change `batchsize`.
3. Use a smoother or rougher starting model.
4. Change the source bandwidth.
5. Plot residual shot records before and after inversion.

## Takeaway

This notebook demonstrates the basic ingredients of single-vintage acoustic FWI:
- a true model,
- a smooth starting model,
- synthetic observed data,
- an adjoint-state gradient through `judiJacobian`,
- and a simple constrained optimization loop.

In practice, robust FWI often uses:
- frequency continuation,
- multiscale strategies,
- source encoding or batching,
- better regularization,
- and more careful parameterization.